# 0503 · 01 · Columnas de metadatos durante la ingesta

En esta demostración vas a añadir columnas de metadatos mientras ingieres datos.

El proceso incluye añadir metadatos de los ficheros, convertir marcas de tiempo Unix al formato estándar `DATE` y registrar cuándo se ingirió cada fila.

## Objetivos de aprendizaje

Al terminar esta lección serás capaz de:

- Modificar columnas durante la ingesta desde el almacenamiento en la nube a tu tabla Bronze.
- Añadir a la tabla Bronze el momento de la ingesta.
- Usar la columna `_metadata` para extraer metadatos de cada fichero (por ejemplo, el nombre o la hora de modificación) durante la ingesta.

## Antes de empezar

1. Ejecuta **`00_preparar_entorno`** (Run all). Copia los cuatro Parquet a `/Volumes/lab0503/upload/upload/users-historical`.
2. Conecta este notebook a **Serverless**.

Todas las tablas del laboratorio se crean en el schema `lab0503.bronze`:

In [ ]:
%sql
USE CATALOG lab0503;
USE SCHEMA bronze;

SELECT current_catalog() AS catalogo, current_schema() AS schema;

## B. Explorar los ficheros de origen

1. Vas a crear una tabla con el histórico de usuarios a partir de los ficheros Parquet guardados en el volumen `/Volumes/lab0503/upload/upload/users-historical` de Unity Catalog.

   Usa la sentencia `LIST` para ver los ficheros del volumen. Ejecuta la celda y revisa el resultado.

   Fíjate en los valores de la columna **name** que empiezan por **part-**: el volumen contiene varios ficheros **Parquet**, cuatro en este caso.

In [ ]:
%sql
-- Listar los ficheros del volumen users-historical
LIST '/Volumes/lab0503/upload/upload/users-historical';

## C. Añadir columnas de metadatos a la tabla Bronze durante la ingesta

Al ingerir datos en la capa Bronze puedes aplicar transformaciones y, además, recuperar metadatos de los ficheros de entrada con la columna **`_metadata`**.

**`_metadata`** es una columna **oculta**, disponible para todos los formatos de fichero soportados. Para incluirla en el resultado tienes que seleccionarla explícitamente en la consulta que lee el origen: un `SELECT *` no la devuelve.

### Requisitos de la ingesta

Durante la ingesta harás lo siguiente:

1. Convertir la marca de tiempo Unix del Parquet en una columna `DATE`.
2. Incluir el **nombre del fichero de entrada**, para saber de qué fichero en bruto viene cada fila.
3. Incluir la **hora de última modificación** del fichero de entrada.
4. Añadir a la tabla Bronze la **hora de ingesta**.

**Nota:** la columna `_metadata` está disponible en todos los formatos de entrada soportados.

1. Ejecuta la celda de abajo para ver los datos Parquet del volumen.

   Fíjate en que la columna **user_first_touch_timestamp** contiene una marca de tiempo Unix: un número entero, no una fecha legible.

In [ ]:
%sql
SELECT *
FROM read_files(
  '/Volumes/lab0503/upload/upload/users-historical',
  format => 'parquet')
LIMIT 10;

### C1. Convertir la hora Unix al ingerir en Bronze

Los valores de **user_first_touch_timestamp** representan el tiempo en **microsegundos** desde la época Unix (1 de enero de 1970, UTC).

Para crear una columna de fecha legible, usa la función [`from_unixtime()`](https://docs.databricks.com/aws/en/sql/language-manual/functions/from_unixtime). Espera **segundos**, así que antes hay que pasar los microsegundos a segundos dividiendo entre 1.000.000.

1. Ejecuta la consulta y revisa el resultado. Genera una columna nueva, **first_touch_date**, convirtiendo la marca de tiempo Unix en una fecha legible.

   Fíjate en que **first_touch_date** es de tipo `DATE`: `from_unixtime` devuelve texto con fecha y hora, y el `cast` se queda solo con la fecha.

In [ ]:
%sql
-- Convertir la marca de tiempo Unix
SELECT
  *,
  cast(from_unixtime(user_first_touch_timestamp/1000000) AS DATE) AS first_touch_date
  -- Alternativa sin dividir a mano: cast(timestamp_micros(user_first_touch_timestamp) AS DATE)
FROM read_files(
  '/Volumes/lab0503/upload/upload/users-historical',
  format => 'parquet')
LIMIT 10;

> **Ojo con la zona horaria.** `from_unixtime` interpreta el instante en la zona horaria de la sesión (`spark.sql.session.timeZone`). En Serverless es UTC por defecto, la misma en la que están generados estos datos. Con otra zona, un usuario que entró a las 23:30 UTC podría aparecer con la fecha del día siguiente.
>
> La función [`timestamp_micros`](https://docs.databricks.com/aws/en/sql/language-manual/functions/timestamp_micros), en el comentario de la celda, convierte los microsegundos directamente a `TIMESTAMP` sin dividir a mano.

### C2. Añadir metadatos de columna al ingerir

Estos son los metadatos que puedes añadir a la tabla Bronze:

- `_metadata.file_modification_time`: la hora de última modificación del fichero de entrada.
- `_metadata.file_name`: el nombre del fichero de entrada.
- [`current_timestamp()`](https://docs.databricks.com/aws/en/sql/language-manual/functions/current_timestamp): devuelve la hora actual (tipo `TIMESTAMP`) al empezar la consulta. Sirve para registrar cuándo se ingirieron los datos.

Puedes leer más sobre la columna `_metadata` en la [documentación de Databricks](https://docs.databricks.com/aws/en/ingestion/file-metadata-column).

1. Ejecuta la consulta de abajo para añadir estas columnas:
   - **file_modification_time** y **source_file**, a partir de la columna **_metadata**, con los datos del fichero de entrada.
   - **ingestion_time**, que registra el momento exacto de la ingesta.

   Fíjate en que **ingestion_time** tiene **el mismo valor en todas las filas**: `current_timestamp()` se evalúa una sola vez por consulta. Por eso sirve para identificar cada carga.

In [ ]:
%sql
-- Añadir columnas de metadatos
SELECT
  *,
  cast(from_unixtime(user_first_touch_timestamp / 1000000) AS DATE) AS first_touch_date,
  _metadata.file_modification_time AS file_modification_time,      -- Hora de última modificación del fichero de origen
  _metadata.file_name AS source_file,                              -- Nombre del fichero de origen
  current_timestamp() as ingestion_time                            -- Hora de la ingesta
FROM read_files(
  '/Volumes/lab0503/upload/upload/users-historical',
  format => 'parquet')
LIMIT 10;

### C3. Crear la tabla Bronze final

1. Júntalo todo en una sentencia `CTAS` para crear la tabla Delta.

   Ejecuta la celda para crear y ver la tabla nueva, **historical_users_bronze**.

   Confirma que la tabla Bronze tiene las columnas nuevas **first_touch_date**, **file_modification_time**, **source_file** e **ingestion_time**. También verás **`_rescued_data`**: la añade `read_files` para recoger datos que no encajan en el esquema y aquí vale `NULL`.

   `CREATE OR REPLACE` permite repetir la celda: cada ejecución reconstruye la tabla desde los ficheros, con un `ingestion_time` nuevo.

In [ ]:
%sql
-- Crear la tabla con CTAS
CREATE OR REPLACE TABLE historical_users_bronze AS
SELECT
  *,
  cast(from_unixtime(user_first_touch_timestamp / 1000000) AS DATE) AS first_touch_date,
  _metadata.file_modification_time AS file_modification_time,      -- Hora de última modificación del fichero de origen
  _metadata.file_name AS source_file,                              -- Nombre del fichero de origen
  current_timestamp() as ingestion_time                            -- Hora de la ingesta
FROM read_files(
  '/Volumes/lab0503/upload/upload/users-historical',
  format => 'parquet');

-- Ver la tabla Bronze final
SELECT *
FROM historical_users_bronze
LIMIT 10;

### C4. Explorar la tabla Bronze final

1. Con las columnas de metadatos en la tabla Bronze puedes explorar de dónde viene cada fila. Por ejemplo, ejecuta una consulta para ver cuántas filas vienen de cada fichero Parquet agrupando por la columna **source_file**.

   Deberías obtener esto, 251.501 filas en total:

   | source_file | total |
   |---|---|
   | part-00000.snappy.parquet | 62.876 |
   | part-00001.snappy.parquet | 62.875 |
   | part-00002.snappy.parquet | 62.875 |
   | part-00003.snappy.parquet | 62.875 |

In [ ]:
%sql
-- Contar filas por fichero Parquet
SELECT
  source_file,
  count(*) as total
FROM historical_users_bronze
GROUP BY source_file
ORDER BY source_file;

## D. (BONUS) Equivalente en Python

La misma ingesta con la API de DataFrames. Con `spark.read`, `_metadata` también es una columna oculta: se pide por su nombre con `col("_metadata.<campo>")`.

Al comparar las dos tablas, verás que la de Python **no tiene `_rescued_data`**. Esa columna es propia de `read_files`, no de `spark.read`.

In [ ]:
from pyspark.sql.functions import col, from_unixtime, current_timestamp
from pyspark.sql.types import DateType

# 1. Leer los Parquet del almacenamiento en un DataFrame de Spark
df = (spark
      .read
      .format("parquet")
      .load("/Volumes/lab0503/upload/upload/users-historical")
    )


# 2. Añadir las columnas de metadatos
df_with_metadata = (
    df.withColumn("first_touch_date", from_unixtime(col("user_first_touch_timestamp") / 1_000_000).cast(DateType()))
      .withColumn("file_modification_time", col("_metadata.file_modification_time"))
      .withColumn("source_file", col("_metadata.file_name"))
      .withColumn("ingestion_time", current_timestamp())
)


# 3. Guardar como tabla Delta
(df_with_metadata
 .write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("lab0503.bronze.historical_users_bronze_python_metadata")
)


# 4. Leer y mostrar la tabla
historical_users_bronze_python_metadata = spark.table("lab0503.bronze.historical_users_bronze_python_metadata")

display(historical_users_bronze_python_metadata)

## Comprueba lo que has aprendido

1. **¿Por qué no aparece `_metadata` con un `SELECT *`?** Es una columna oculta: hay que pedirla explícitamente, por ejemplo `_metadata.file_name`.
2. **¿Por qué se divide entre 1.000.000?** `user_first_touch_timestamp` está en microsegundos y `from_unixtime` espera segundos.
3. **¿Por qué todas las filas tienen el mismo `ingestion_time`?** `current_timestamp()` se calcula una vez al empezar la consulta, no fila a fila.
4. **¿Para qué sirve `source_file` en Bronze?** Para trazar cada fila hasta su fichero de origen: auditar, depurar un fichero defectuoso o comprobar que no falta ninguno, como en la consulta C4.
5. **¿Qué diferencia hay entre `file_modification_time` e `ingestion_time`?** El primero es cuándo cambió el fichero en el almacenamiento; el segundo, cuándo lo cargaste en la tabla.

## Volver a empezar

Ejecuta `00_preparar_entorno`: borra las dos tablas y deja los Parquet como al principio. Las tablas no se borran al terminar este notebook, para que puedas seguir inspeccionándolas.

## Recursos adicionales

- [Columna `_metadata` de ficheros](https://docs.databricks.com/aws/en/ingestion/file-metadata-column)
- [read_files](https://docs.databricks.com/aws/en/sql/language-manual/functions/read_files)
- [from_unixtime](https://docs.databricks.com/aws/en/sql/language-manual/functions/from_unixtime)
- [timestamp_micros](https://docs.databricks.com/aws/en/sql/language-manual/functions/timestamp_micros)
- [current_timestamp](https://docs.databricks.com/aws/en/sql/language-manual/functions/current_timestamp)